### Panel VAR
Ce notebook utilise le panel `full_panel_interactions`, teste la stationnarité des séries individuelles, détermine le nombre de lags optimal, estime un PVAR, et calcule les IRFs avec une décomposition de Cholesky.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt

### 0. Importation

In [ ]:
df = pd.read_csv("../Reddit-Polarization/clean_data/full_panel_interactions.csv")
df['period'] = pd.to_datetime(df['period'], format='%Y-%m')
df = df.sort_values(['user', 'period'])

### 1. Préparation
Filtrer le dataframe pour obtenir des séries assez longues
Et statistiques sur la couverture et le déséquilibre du panel

In [ ]:
def imbalance(df):
    # Déséquilibre
    n_users = df['user'].nunique()
    print('Users : ', n_users)
    n_periods = df['period'].nunique()
    total_possible = n_users * n_periods
    total_actual = len(df)
    print("Déséquilibre :", (1 - total_actual/total_possible)*100)

    # Distribution des longueurs de séries
    ts_lengths = df.groupby('user')['period'].count()
    q25 = ts_lengths.quantile(0.25)
    q50 = ts_lengths.quantile(0.50)
    q75 = ts_lengths.quantile(0.75)
    print(f"Min : {ts_lengths.min()}")
    print(f"Q1 : {q25}")
    print(f"Q2 : {q50}")
    print(f"Q3 : {q75}")
    print(f"Max : {ts_lengths.max()}")

In [ ]:
imbalance(df)
print(len(df))

# Filtrer les individus ayant au moins 6 observations non manquantes
k = 6
df_filtered = (
    df.groupby("user")
      .filter(lambda x: x["period"].notna().sum() >= k)
)

imbalance(df_filtered)
print(len(df_filtered))

In [ ]:
df = df_filtered

### 2. Tests de stationnarité

In [ ]:
def adf_manual(series, lags=1):
    s = np.array(series, dtype=float)
    T = len(s)
    if T < lags + 3:
        return np.nan, np.nan
    
    dy = np.diff(s)
    rows = []
    for t in range(lags, len(dy)):
        row = [s[t]]  # y_{t-1} (niveau)
        for l in range(1, lags + 1):
            row.append(dy[t - l])  # Δy_{t-l}
        row.append(1.0)  # constante
        rows.append(row)
    
    X = np.array(rows)
    y = dy[lags:]
    
    if len(y) < 3 or X.shape[0] < X.shape[1]:
        return np.nan, np.nan
    
    try:
        # OLS
        beta, res, rank, sv = np.linalg.lstsq(X, y, rcond=None)
        y_hat = X @ beta
        resid = y - y_hat
        s2 = np.sum(resid**2) / (len(y) - X.shape[1])
        XtX_inv = np.linalg.inv(X.T @ X)
        se = np.sqrt(s2 * np.diag(XtX_inv))
        t_stat = beta[0] / se[0]  # test sur le coefficient de y_{t-1}
        return t_stat, beta[0]
    except:
        return np.nan, np.nan

In [ ]:
# On ne teste que les variables numériques
num_vars = ['n_obs', 'user_total_political_interactions', 'user_polarization', 'user_homophilie']
for var in num_vars:
    t_stats = []
    for user, grp in df.groupby('user'):
        if len(grp) >= 10:  # minimum pour tester
            t, b = adf_manual(grp[var].values, lags=1)
            if not np.isnan(t):
                t_stats.append(t)
    
    t_arr = np.array(t_stats)
    pct_reject = np.mean(t_arr < -2.86) * 100  # % qui rejettent H0
    print(f"{var}:")
    print("N individus testés   : ", len(t_arr))
    print(f"ADF % rejet H0 5%  : ", pct_reject)
    
    # Test de Fisher-type (Im-Pesaran-Shin simplifié) : somme des stats
    N = len(t_arr)
    E_t = -1.54
    V_t = 1.30
    IPS_stat = np.sqrt(N) * (np.mean(t_arr) - E_t) / np.sqrt(V_t)

    print('p-val IPS', norm.cdf(IPS_stat), "\n\n")

### 3. Sélection du meilleur lag

In [ ]:
dummy_vars = ['user_Left', 'user_Right']
all_vars = num_vars + dummy_vars

# Pour la sélection de lag, on se réduit à un sous-panel avec un T uniforme (T=20) par individu 
ts_lengths = df.groupby('user')['period'].count()
users_balanced = ts_lengths[ts_lengths >= 20].index
df_bal = df[df['user'].isin(users_balanced)].copy()
T_ref = 20
balanced_chunks = []
for user, grp in df_bal.groupby('user'):
    balanced_chunks.append(grp.head(T_ref))
df_bal2 = pd.concat(balanced_chunks)

N = df_bal2['user'].nunique()
T = T_ref
K = len(all_vars)

print(f"Sous-panel{N}, {T}\n")

def ols_aic_bic(Y, X):
    T_eff, K = Y.shape
    total_params = 0
    
    residuals = []
    for k in range(K):
        y = Y[:, k]
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        residuals.append(resid)
        total_params += X.shape[1]
    
    Resid = np.array(residuals).T
    Sigma = Resid.T @ Resid / T_eff 
    sign, log_det = np.linalg.slogdet(Sigma)
    
    AIC = log_det + 2 * total_params / T_eff
    BIC = log_det + total_params * np.log(T_eff) / T_eff
    return AIC, BIC

print("Lag&AIC & BIC\\\\\\hline")

results = {}
for p in range(1, 5):
    all_Y = []
    all_X = []
    
    for user, grp in df_bal2.groupby('user'):
        data = grp[all_vars].values.astype(float)
        if len(data) < p + 2:
            continue
        for t in range(p, len(data)):
            y_row = data[t]
            x_row = []
            for lag in range(1, p + 1):
                x_row.extend(data[t - lag])
            x_row.append(1.0)  # constante
            all_Y.append(y_row)
            all_X.append(x_row)
    
    Y = np.array(all_Y)
    X = np.array(all_X)
    
    aic, bic  = ols_aic_bic(Y, X)
    results[p] = (aic, bic)
    print(f"{p} & {aic:.4f} & {bic:.4f}\\\\")

In [ ]:
# Affichage des critères d'information extrapolés par lag
results_df = pd.DataFrame.from_dict(
    results,
    orient='index',
    columns=['AIC', 'BIC']
)

results_df.index.name = 'Lag'
results_df = results_df.reset_index()

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif'
})

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    results_df['Lag'],
    results_df['AIC'],
    marker='o',
    linewidth=2,
    markersize=7,
    label='AIC'
)

ax.plot(
    results_df['Lag'],
    results_df['BIC'],
    marker='s',
    linewidth=2,
    markersize=7,
    label='BIC'
)

ax.set_xlabel('Nombre de lags')
ax.set_ylabel("Critère d'information")
ax.set_title('Sélection du nombre de lags du PVAR')

ax.set_xticks(results_df['Lag'])
ax.grid(True, linestyle='--', alpha=0.4)

ax.legend(frameon=False)

plt.tight_layout()

plt.show()


On choisit finalement un lag $p=2$

### 4. Transformation de Helmert

In [ ]:
K = len(all_vars)
p = 2 

def helmert_transform(grp, variables):
    data = grp[variables].values.astype(float)
    T = len(data)
    transformed = []
    for t in range(T - 1):  # on perd la dernière obs
        future_mean = data[t+1:].mean(axis=0)
        weight = np.sqrt((T - t - 1) / (T - t))
        transformed.append(weight * (data[t] - future_mean))
    result = grp.iloc[:-1].copy()
    for j, var in enumerate(variables):
        result[var + '_helm'] = [row[j] for row in transformed]
    return result

helm_chunks = []
for user, grp in df.groupby('user'):
    if len(grp) >= p + 2:  # T minimum pour avoir des obs après Helmert + lags
        h = helmert_transform(grp, all_vars)
        helm_chunks.append(h)

df_helm = pd.concat(helm_chunks).reset_index(drop=True)
helm_vars = [v + '_helm' for v in all_vars]

print(f"i x t après transformation Helmert: {len(df_helm)}")
print('Verif', len(df_filtered)-df_helm['user'].nunique())
print(f"Utilisateurs : {df_helm['user'].nunique()}")

### 5. Construction des matrices Y et X

In [ ]:
df_levels = df.copy()
for var in all_vars:
    for lag in range(1, p + 1):
        df_levels[f'{var}_L{lag}'] = df_levels.groupby('user')[var].shift(lag)

# Merger les valeurs Helmert avec les lags des niveaux
df_merged = df_helm[['user', 'period'] + helm_vars].merge(
    df_levels[['user', 'period'] + [f'{v}_L{l}' for v in all_vars for l in range(1, p+1)]],
    on=['user', 'period'],
    how='left'
)
df_merged = df_merged.dropna()

print(f"i x t merge avec lags: {len(df_merged)}")
print('verif', len(df_helm)-2*df_merged['user'].nunique())
print(f"Utilisateurs : {df_merged['user'].nunique()}")

### 6. Estimation OLS

In [ ]:
Y = df_merged[helm_vars].values
lag_cols = [f'{v}_L{l}' for l in range(1, p+1) for v in all_vars]
X = df_merged[lag_cols].values

coef_matrix = np.zeros((K, K * p))
se_matrix = np.zeros((K, K * p))
t_matrix = np.zeros((K, K * p))
resid_matrix = np.zeros((Y.shape[0], K))

for k in range(K):
    y_k = Y[:, k]
    beta, _, _, _ = np.linalg.lstsq(X, y_k, rcond=None)
    resid = y_k - X @ beta
    resid_matrix[:, k] = resid
    
    T_eff = len(y_k)
    n_params = X.shape[1]
    s2 = np.sum(resid**2) / (T_eff - n_params)
    
    try:
        cov_beta = s2 * np.linalg.inv(X.T @ X)
        se = np.sqrt(np.diag(cov_beta))
    except:
        se = np.full(n_params, np.nan)
    
    t_stats = beta / se
    coef_matrix[k] = beta
    se_matrix[k] = se
    t_matrix[k] = t_stats

# Matrice de covariance des résidus
Sigma = resid_matrix.T @ resid_matrix / len(resid_matrix)
meta = {'all_vars': all_vars, 'p': p, 'K': K, 'lag_cols': lag_cols, 
                 'N_eff': df_merged['user'].nunique(), 'T_obs': len(df_merged)}

A = []
for lag in range(1, p + 1):
    A_lag = coef_matrix[:, (lag-1)*K : lag*K]
    A.append(A_lag)

### 7. IRF
En raison du bootstrap pour calculer les intervalles de confiance la réplication peut être longue (jusqu'à 1h pour nous.)

In [ ]:
A_matrices = np.array(A)

# Décomposition de Cholesky pour identification
# Ordre de Cholesky : n_obs → total_interactions → polarization → homophilie → Left → Right
P = np.linalg.cholesky(Sigma)

In [ ]:
def compute_irf(A_matrices, P, n_ahead=12):
    """
    Calcule les IRF orthogonalisées.
    A_matrices : (p, K, K) - matrices de coefficients
    P : (K, K) - facteur de Cholesky de Sigma
    Retourne : (K, K, n_ahead+1) - irf[choc, variable, horizon]
    """
    K = A_matrices.shape[1]
    p = A_matrices.shape[0]
    
    Phi = np.zeros((n_ahead + 1, K, K))
    Phi[0] = np.eye(K)
    
    for h in range(1, n_ahead + 1):
        for j in range(1, min(h, p) + 1):
            Phi[h] += A_matrices[j-1] @ Phi[h-j]
    
    Theta = np.zeros((n_ahead + 1, K, K))
    for h in range(n_ahead + 1):
        Theta[h] = Phi[h] @ P
    
    return Theta

def helmert_transform_fast(data):
    T = len(data)
    transformed = np.zeros((T-1, data.shape[1]))
    for t in range(T-1):
        future_mean = data[t+1:].mean(axis=0)
        weight = np.sqrt((T - t - 1) / (T - t))
        transformed[t] = weight * (data[t] - future_mean)
    return transformed

def pvar_from_data(df_input, all_vars, p):
    """Estime un PVAR complet et retourne les IRF"""
    K = len(all_vars)
    
    # Helmert + construction Y, X
    all_Y_list, all_X_list = [], []
    
    for user, grp in df_input.groupby('user'):
        data = grp[all_vars].values.astype(float)
        if len(data) < p + 2:
            continue
        helm = helmert_transform_fast(data)
        T_h = len(helm)
        
        # Lags des niveaux (pas de la transformation)
        for t in range(p, T_h):
            y_row = helm[t]
            x_row = []
            for lag in range(1, p+1):
                # Index dans les données originales : t - lag (la série helm[t] correspond à data[t])
                x_row.extend(data[t - lag])
            all_Y_list.append(y_row)
            all_X_list.append(x_row)
    
    if len(all_Y_list) < K * p + 10:
        return None
    
    Y_b = np.array(all_Y_list)
    X_b = np.array(all_X_list)
    
    # OLS
    A_b = np.zeros((p, K, K))
    resid_b = np.zeros((len(Y_b), K))
    
    for k in range(K):
        beta, _, _, _ = np.linalg.lstsq(X_b, Y_b[:, k], rcond=None)
        resid_b[:, k] = Y_b[:, k] - X_b @ beta
        for lag in range(p):
            A_b[lag, k, :] = beta[lag*K:(lag+1)*K]
    
    Sigma_b = resid_b.T @ resid_b / len(resid_b)
    
    try:
        P_b = np.linalg.cholesky(Sigma_b + np.eye(K) * 1e-10)
    except:
        return None
    
    return compute_irf(A_b, P_b, n_ahead)


def plot_irf_matrix(Theta, irf_lower, irf_upper, n_ahead, var_names=None):
    K = Theta.shape[2]
    h = np.arange(n_ahead + 1)

    fig, axes = plt.subplots(K, K, figsize=(3*K, 3*K), sharex=True)

    for i in range(K):        # réponse (ligne)
        for j in range(K):    # choc (colonne)

            ax = axes[i, j]

            irf = Theta[:, i, j]
            lower = irf_lower[:, i, j]
            upper = irf_upper[:, i, j]

            ax.plot(h, irf, color="black", linewidth=1.5)
            ax.fill_between(h, lower, upper, alpha=0.3)

            ax.axhline(0, color="black", linewidth=0.8)
            ax.axvline(0, color="black", linewidth=0.5)

            # labels seulement sur bords (style papier)
            if i == K - 1:
                ax.set_xlabel(f"Shock {var_names[j] if var_names else j}")
            if j == 0:
                ax.set_ylabel(f"Resp {var_names[i] if var_names else i}")

            ax.tick_params(labelsize=6)

    plt.tight_layout()
    plt.show()

In [ ]:
n_ahead = 12
Theta = compute_irf(A_matrices, P, n_ahead)

# Bootstrap
users = df['user'].unique()
N = len(users)

boot_irfs = []
np.random.seed(42)

for b in range(500):
    # Tirage avec remise des individus
    sampled_users = np.random.choice(users, size=N, replace=True)
    boot_chunks = []
    for u in sampled_users:
        boot_chunks.append(df[df['user'] == u].copy())
    df_boot = pd.concat(boot_chunks).reset_index(drop=True)
    # Renommer les users pour éviter les doublons dans groupby
    df_boot['user'] = df_boot.groupby('user').ngroup().astype(str) + '_' + \
                      df_boot.groupby(level=0).cumcount().astype(str)
    # Solution alternative : assigner un user_id unique
    user_map = {u: f"u{i}" for i, u in enumerate(sampled_users)}
    df_boot2 = []
    for i, u in enumerate(sampled_users):
        chunk = df[df['user'] == u].copy()
        chunk['user'] = f"u{i}"
        df_boot2.append(chunk)
    df_boot2 = pd.concat(df_boot2)
    
    theta_b = pvar_from_data(df_boot2, all_vars, p)
    if theta_b is not None:
        boot_irfs.append(theta_b)

boot_irfs = np.array(boot_irfs)  # (n_boot, n_ahead+1, K, K)
irf_lower = np.percentile(boot_irfs, 5, axis=0)
irf_upper = np.percentile(boot_irfs, 95, axis=0)

In [ ]:
plot_irf_matrix(
    Theta,
    irf_lower,
    irf_upper,
    n_ahead=n_ahead,
    var_names=all_vars
)